# AP5 — Agentic AI, LangChain, LangGraph et MCP (version autonome)
**Module :** IA Générative Multi Modale et Agentique  
**Étudiante :** Salma Riad

Ce notebook **ne dépend pas d’un dossier `src/`** : les démonstrations métier, outils et tests locaux sont inclus. Les parties LangChain + Ollama, LangGraph et MCP sont de **vraies intégrations optionnelles**, seulement exécutées si leurs dépendances et services sont disponibles. Les données et notices PX-200 sont fictives : ce TP ne vérifie aucune garantie réelle.

## 0 — Préparation

Le **mode démonstration locale** utilise seulement Python 3.11+ (bibliothèque standard). Il permet de tester directement le notebook sous Python 3.13. Pour activer les parties avancées, installer séparément dans le kernel Jupyter :

```powershell
python -m pip install langgraph langchain langchain-ollama mcp ipykernel
```

Pour l'agent LLM, démarrer Ollama et récupérer `llama3.2`. Le notebook n'installe rien automatiquement, ne démarre aucun serveur et n'utilise aucune clé API.

In [ ]:
import sys, re, importlib.util, json
from dataclasses import dataclass
from datetime import date
from typing import TypedDict
print("Python :", sys.version.split()[0])
for package in ["langgraph", "langchain", "langchain_ollama", "mcp"]:
    print(f"{package}: {'disponible' if importlib.util.find_spec(package) else 'facultatif / non installé'}")

## A — Agents et outils, démonstration locale autonome

Un outil de recherche documentaire et un outil de consultation de garantie simulée. Les **vrais agents LLM** seront montrés dans la section optionnelle. Ici, le routage reste une règle explicite, **pas** un raisonnement autonome du LLM.

In [ ]:
DOCUMENTS = [
    {"source": "manuel_demo_PX200.txt", "page": 1, "text": "PX-200 : un voyant orange clignotant peut signaler une anomalie de papier. Vérifier le bac papier. Cette notice est fictive."},
    {"source": "manuel_demo_PX200.txt", "page": 2, "text": "PX-200 : un pictogramme d'entretien invite à consulter le manuel du constructeur. Exemple fictif."},
]
GARANTIES_DEMO = {"PX200-0042": {"statut": "simulée active", "expiration": "2027-02-01"}}

def rechercher_manuel(question, k=3):
    mots = set(re.findall(r"[a-zà-ÿ0-9-]{4,}", question.lower())) - {"mon", "avec", "dans", "pour", "est", "une"}
    resultats = []
    for passage in DOCUMENTS:
        score = sum(m in passage["text"].lower() for m in mots)
        if score:
            resultats.append((score, passage))
    return [p for _, p in sorted(resultats, key=lambda x: -x[0])[:k]]

def consulter_garantie(numero):
    numero = numero.strip().upper()
    if not re.fullmatch(r"[A-Z0-9-]{4,40}", numero):
        return {"numero": numero, "statut": "numéro invalide", "source": "simulation"}
    info = GARANTIES_DEMO.get(numero, {"statut": "inconnue"})
    return {"numero": numero, **info, "source": "simulation locale, non officielle"}

print(rechercher_manuel("Mon imprimante PX-200 a un voyant orange clignotant"))
print(consulter_garantie("PX200-0042"))

In [ ]:
def router(question):
    return "garantie" if re.search(r"garantie|numéro de série|numero de serie|warranty", question, re.I) else "documentation"

def repondre(question):
    categorie = router(question)
    if categorie == "garantie":
        numeros = re.findall(r"\b[A-Za-z]{2,}\d{2,}-\d{3,}\b", question)
        if not numeros:
            return {"agent": "agent_garantie", "reponse": "Indiquez un numéro de série pour la démonstration.", "sources": []}
        return {"agent": "agent_garantie", "reponse": consulter_garantie(numeros[0]), "sources": []}
    passages = rechercher_manuel(question)
    if not passages:
        return {"agent": "agent_documentaire", "reponse": "Aucun passage trouvé ; réponse non confirmée.", "sources": []}
    return {"agent": "agent_documentaire", "reponse": " ".join(p["text"] for p in passages), "sources": [{"document": p["source"], "page": p["page"]} for p in passages]}

for question in ["Mon imprimante PX-200 a un voyant orange clignotant", "Mon PX200-0042 est encore sous garantie ?"]:
    print(json.dumps(repondre(question), ensure_ascii=False, indent=2))

### Mémoire locale de conversation
Cette mémoire simple stocke l’historique en mémoire Python. Elle n’est pas équivalente à un *checkpointer* LangGraph ni à la mémoire d’un agent LLM.

In [ ]:
HISTORIQUE = []
def discuter(question):
    resultat = repondre(question)
    HISTORIQUE.append({"question": question, "resultat": resultat})
    return resultat
print(discuter("Que signifie le voyant orange PX-200 ?"))
print("Échanges enregistrés :", len(HISTORIQUE))

### Agent LangChain + mémoire : option réelle (Ollama requis)
Quand les paquets sont installés et Ollama disponible, cette cellule construit un agent `create_agent` utilisant de vrais appels d'outils et un checkpointer. Par défaut, elle ne démarre pas le LLM.

In [ ]:
EXECUTER_AGENT_LLM = False
if EXECUTER_AGENT_LLM:
    try:
        from langchain.agents import create_agent
        from langchain.tools import tool
        from langchain_ollama import ChatOllama
        from langgraph.checkpoint.memory import InMemorySaver
        @tool
        def rechercher_notice(question: str) -> str:
            """Chercher des extraits dans la notice fictive PX-200."""
            return json.dumps(rechercher_manuel(question), ensure_ascii=False)
        @tool
        def verifier_garantie(numero: str) -> str:
            """Vérifier une garantie fictive de démonstration à partir d'un numéro."""
            return json.dumps(consulter_garantie(numero), ensure_ascii=False)
        agent = create_agent(
            ChatOllama(model="llama3.2", temperature=0),
            tools=[rechercher_notice, verifier_garantie],
            checkpointer=InMemorySaver(),
            system_prompt="Assistant SAV de démonstration. Ne présente jamais les garanties fictives comme officielles. Cite les extraits disponibles.",
        )
        resultat = agent.invoke({"messages": [{"role": "user", "content": "Quel est le sens du voyant orange PX-200 ?"}]},
                                config={"configurable": {"thread_id": "tp5-salma"}})
        print(resultat["messages"][-1].content)
    except Exception as erreur:
        print("Agent réel indisponible :", type(erreur).__name__, str(erreur)[:350])
else:
    print("Mode local : agent LangChain/Ollama non lancé (option facultative).")

## B — LangGraph : orchestrateur de deux agents spécialisés
Si LangGraph est disponible dans le kernel, le graphe est compilé et exécuté. Sinon, la simulation locale déjà testée reste utilisable, mais n’est **pas** appelée LangGraph.

In [ ]:
if importlib.util.find_spec("langgraph"):
    from langgraph.graph import StateGraph, START, END
    class Etat(TypedDict, total=False):
        question: str
        categorie: str
        agent: str
        reponse: object
        sources: list
    def categoriser(etat):
        return {"categorie": router(etat["question"])}
    def agent_documentaire(etat):
        return repondre(etat["question"])
    def agent_garantie(etat):
        return repondre(etat["question"])
    graphe = StateGraph(Etat)
    graphe.add_node("orchestrateur", categoriser)
    graphe.add_node("agent_documentaire", agent_documentaire)
    graphe.add_node("agent_garantie", agent_garantie)
    graphe.add_edge(START, "orchestrateur")
    graphe.add_conditional_edges("orchestrateur", lambda e: "agent_garantie" if e["categorie"] == "garantie" else "agent_documentaire",
                                 {"agent_garantie": "agent_garantie", "agent_documentaire": "agent_documentaire"})
    graphe.add_edge("agent_documentaire", END)
    graphe.add_edge("agent_garantie", END)
    application = graphe.compile()
    for question in ["Voyant orange PX-200", "Garantie PX200-0042"]:
        print(application.invoke({"question": question}))
else:
    print("LangGraph non installé : intégration sautée ; démonstration locale disponible plus haut.")

## C — MCP : création du serveur à deux outils
Cette cellule **construit** un serveur MCP quand le SDK est présent. Elle ne le démarre pas dans le notebook, pour éviter de bloquer le kernel. Pour le tester avec MCP Inspector ou un client streamable-HTTP, il faut lancer un serveur séparé (voir README du dossier).

In [ ]:
if importlib.util.find_spec("mcp"):
    from mcp.server.fastmcp import FastMCP
    serveur_mcp = FastMCP("tp5-sav-demo")
    @serveur_mcp.tool()
    def chercher_manuel(question: str) -> str:
        """Rechercher des extraits dans une notice de démonstration."""
        return json.dumps(rechercher_manuel(question), ensure_ascii=False)
    @serveur_mcp.tool()
    def garantie_demo(numero: str) -> str:
        """Consulter le statut fictif de garantie, jamais une base officielle."""
        return json.dumps(consulter_garantie(numero), ensure_ascii=False)
    print("Serveur MCP construit : chercher_manuel et garantie_demo ; transport non démarré.")
else:
    print("SDK MCP non installé : serveur non construit. Les fonctions locales sont opérationnelles.")

## D — Synthèse : SAV avec orchestrateur
Dans l’énoncé, l’objectif est un assistant SAV avec documentation RAG multimodale, garantie via MCP et orchestration LangGraph. **Ici** le scénario local est une preuve de concept *textuelle* avec extraits fictifs. Il ne contient pas de vrai PDF, ni de recherche image, ni de connexion n8n. Ces capacités doivent être intégrées et validées séparément.

In [ ]:
questions = [
    "Mon imprimante PX-200 affiche un voyant orange clignotant : que signifie-t-il ?",
    "Le numéro PX200-0042 est-il encore sous garantie ?",
]
for q in questions:
    print("QUESTION :", q)
    print("RÉSULTAT :", json.dumps(repondre(q), ensure_ascii=False))
    print()

## Tests et bilan honnête
Les assertions suivantes testent le module local directement dans le notebook. La réussite de ces assertions **ne prouve pas** le fonctionnement d'Ollama, de MCP Inspector, de n8n ou d'un agent RAG multimodal réel.

In [ ]:
assert rechercher_manuel("voyant orange PX-200"), "Recherche documentaire sans résultat"
assert consulter_garantie("PX200-0042")["statut"] == "simulée active"
assert consulter_garantie("! invalide")["statut"] == "numéro invalide"
assert router("Est-il encore sous garantie ?") == "garantie"
assert router("Voyant orange sur mon imprimante") == "documentation"
assert repondre("PX200-0042 sous garantie ?")["agent"] == "agent_garantie"
assert repondre("Un sujet totalement absent : volcan lunaire")["sources"] == []
print("7/7 tests locaux réussis (bibliothèque standard).")

## À valider pour la remise
- Tester l’agent LangChain + Ollama (vrais appels LLM et outils, mémoire persistante durant une session).
- Vérifier les variantes de middleware : modèle/prompt dynamiques, gestion d’erreurs, garde-fous et intervention humaine.
- Tester LangGraph avec le SDK installé et, si nécessaire, LangGraph Studio.
- Lancer le serveur MCP séparément et vérifier ses outils dans MCP Inspector et depuis un client MCP.
- Réaliser les demandes de synthèse avec **vrais documents PDF/images**, véritable outil de garantie et agent n8n, si demandés.

L’objectif du notebook autonome est de supprimer l’erreur `No module named 'src'` et d’avoir un exemple **exécutable et vérifiable localement**, sans prétendre que toutes les intégrations externes sont validées.